In [55]:
from datasets import load_dataset
dataset = list(load_dataset("lmarena-ai/arena-human-preference-100k")["train"])

In [56]:
print(f"Total number of samples: {len(dataset)}")
dataset_english = [d for d in dataset if d["language"] == "English"]
print(f"Number of English samples: {len(dataset_english)}")

Total number of samples: 106134
Number of English samples: 57675


In [ ]:
from matplotlib import pyplot as plt
from collections import Counter
import json

dataset_en_creative = [d for d in dataset_english if d["category_tag"]["criteria_v0.1"]["creativity"] and d["turn"] == 1]
print(f"Number of creative samples: {len(dataset_en_creative)}")
# dataset_en_creative[0]

print(Counter(d['winner'] for d in dataset_en_creative))

dataset_non_ties = [d for d in dataset_en_creative if d['winner'] in ["model_a", "model_b"]]
print(f"Number of non-ties: {len(dataset_non_ties)}")

dataset_short_length = [d for d in dataset_non_ties if 100 <= d['conversation_a'][-1]['num_tokens'] < 300 and 100 <= d['conversation_b'][-1]['num_tokens'] < 300]
print(f"Number of short length samples: {len(dataset_short_length)}")

# plot a histogram of the conversation_a response in number of words and conversation_b response in number of words
# plt.figure(figsize=(10, 5))
# plt.hist([d['conversation_a'][-1]['num_tokens'] for d in dataset_non_ties], bins=50, alpha=0.5, label='conversation_a')
# plt.hist([d['conversation_b'][-1]['num_tokens'] for d in dataset_non_ties], bins=50, alpha=0.5, label='conversation_b')
# plt.legend()
# plt.show()

with open("data/lmarena_filtered_2k_pairwise_balanced.json", "w") as f:
    json.dump(dataset_short_length, f)


Number of creative samples: 15198
Counter({'model_b': 5156, 'model_a': 4852, 'tie': 2719, 'tie (bothbad)': 2471})
Number of non-ties: 10008
Number of short length samples: 1188


In [1]:
from collections import Counter
from utils import load_env_vars
load_env_vars()

from llms import generate_json
import tqdm, json

prompt_creative_writing_instruction = """You are given an instruction that a user provided to an LLM service.
Your objective is to classify whether yes/no the instruction is a creative writing task.

A creative writing task is defined as an instruction that asks an LLM to produce writing for a creative purpose, such as a story, poem, essay, etc.

Here are examples of instructions that are *NOT* creative writing tasks:
- Asking an LLM to create a riddle, crossword,
- Asking an LLM to come up with a joke

Rules:
- You must generate your response as a JSON object, in the following format: {"is_creative_writing_task": "yes" | "no", "reason": "explanation of your answer"}
- Keep your reason concise, below 10 words.

Here is the instruction:
[[INSTRUCTION]]
"""

with open("data/lmarena_filtered_2k_pairwise_balanced.json", "r") as f:
    dataset_short_length = json.load(f)

yes_no_counts = Counter()
ite = tqdm.tqdm_notebook(dataset_short_length)
for d in ite:
    if "creative_cls_gpt4o" in d:
        continue
    instruction = d['conversation_a'][0]['content']
    populated_prompt = prompt_creative_writing_instruction.replace("[[INSTRUCTION]]", instruction)
    resp = generate_json([{"role": "user", "content": populated_prompt}], model="gpt-4o")
    # if it's a yes, print in green + the response, else print in red + the response
    # print("---------------------------------------")
    # if resp['is_creative_writing_task'] == "yes":
    #     print(f"\033[92m{instruction}\033[0m")
    #     print(f"\033[92m{resp['reason']}\033[0m")
    # else:
    #     print(f"\033[91m{instruction}\033[0m")
    #     print(f"\033[91m{resp['reason']}\033[0m")
    d['creative_cls_gpt4o'] = resp
    yes_no_counts[resp['is_creative_writing_task']] += 1
    ite.set_description(f"Yes: {yes_no_counts['yes']}, No: {yes_no_counts['no']}")

    with open("data/lmarena_filtered_2k_pairwise_balanced.json", "w") as f:
        json.dump(dataset_short_length, f)


/tmp/ipykernel_1382732/383773900.py:29: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  ite = tqdm.tqdm_notebook(dataset_short_length)


  0%|          | 0/1188 [00:00<?, ?it/s]

In [4]:
with open("data/lmarena_filtered_2k_pairwise_balanced.json", "r") as f:
    dataset_short_length = json.load(f)

with open("prompts/pairwise_pref.txt", "r") as f:
    pairwise_prompt = f.read()

verified_dataset = [d for d in dataset_short_length if "creative_cls_gpt4o" in d and d['creative_cls_gpt4o']['is_creative_writing_task'] == "yes"]

final_dataset = []
for didx, d in enumerate(verified_dataset):
    sample1 = {"id": f"test-lmarena-{len(final_dataset)}", "original_id": d['question_id'], "split": "test", "sample_type": "pairwise-lmarena", "model_a": d['model_a'], "model_b": d['model_b']}
    para1, para2 = d['conversation_a'][1]["content"], d['conversation_b'][1]["content"]
    sample1["user_instruction"] = d['conversation_a'][0]["content"]
    sample1["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", para1).replace("[[PARAGRAPH2]]", para2)
    sample1["paragraph1"] = para1
    sample1["paragraph2"] = para2
    sample1["reference_preference"] = "1" if d['winner'] == "model_a" else "2"
    final_dataset.append(sample1)

    sample2 = {"id": f"test-lmarena-{len(final_dataset)}", "original_id": d['question_id'], "split": "test", "sample_type": "pairwise-lmarena", "model_a": d['model_b'], "model_b": d['model_a']}
    para1, para2 = d['conversation_b'][1]["content"], d['conversation_a'][1]["content"]
    sample2["user_instruction"] = d['conversation_b'][0]["content"]
    sample2["text_input"] = pairwise_prompt.replace("[[PARAGRAPH1]]", para1).replace("[[PARAGRAPH2]]", para2)
    sample2["paragraph1"] = para1
    sample2["paragraph2"] = para2
    sample2["reference_preference"] = "2" if d['winner'] == "model_a" else "1"
    final_dataset.append(sample2)

print(len(final_dataset))
with open("data/lamp_PRGSH_test.json", "r") as f:
    lamp_PRGSH_test = json.load(f)

lamp_PRGSH_test = [d for d in lamp_PRGSH_test if d["sample_type"] != "pairwise-lmarena"] # remove what was previously added

lamp_PRGSH_test += final_dataset

with open("data/lamp_PRGSH_test.json", "w") as f:
    json.dump(lamp_PRGSH_test, f, indent=4)

print(Counter(d['sample_type'] for d in lamp_PRGSH_test))

576
Counter({'pairwise-gold': 1206, 'pairwise-silver': 1120, 'pairwise-lmarena': 576, 'reward': 430, 'pairwise': 404, 'pairwise-h': 300, 'pairwise-P1': 215, 'pairwise-P2': 215, 'pairwise-P3': 209, 'pairwise-P4': 199, 'pairwise-P5': 183, 'pairwise-P6': 159, 'pairwise-P7': 138})


In [6]:
with open("data/lamp_PRGSH_test.json", "r") as f:
    lamp_PRGSH_test = json.load(f)

print(Counter(d['sample_type'] for d in lamp_PRGSH_test))

Counter({'pairwise-gold': 1206, 'pairwise-silver': 1120, 'pairwise-lmarena': 576, 'reward': 430, 'pairwise': 404, 'pairwise-h': 300, 'pairwise-P1': 215, 'pairwise-P2': 215, 'pairwise-P3': 209, 'pairwise-P4': 199, 'pairwise-P5': 183, 'pairwise-P6': 159, 'pairwise-P7': 138})
